# Phase 5 - Legal Statute Identification: all experiments in one notebook

**Run All** runs every experiment one after another. Everything is cached: a finished training is loaded, finished
reports / predictions / zips are skipped, and an interrupted training resumes from its last checkpoint
(every 250 optimiser steps). So you can stop and re-run this notebook at any time.

Pipeline (same for every experiment, `src/lsi_utils.py`, `src/lsi_models.py`, `src/lsi_experiment.py`):

    case text -> Phase-1 cleaning (cased, placeholders <ENTITY>/<SECTION>/<ACT> kept) -> bpe32k tokens
              -> first 768 + last 768 tokens (D: 512 + 512)
              -> Phase-1 backbone -> attention pooling -> Linear(640 -> 100) -> sigmoid = P(statute l applies)
    loss  = BCE per statute with pos_weight = (neg/pos) ** 0.5
    dev   = best epoch (macro-F1 at the best global threshold) + thresholds: one global vs one per statute,
            compared honestly by 2-fold cross-fitting on dev; every case keeps >= 1 statute
    test  = statute sets with the chosen thresholds -> submission.zip in the run folder

Decisions from P5_01 (PROJECT_NOTES 12.3): train duplicates removed and conflicting duplicates dropped; dev/test untouched;
no lower-casing; first + last tokens for the flat models; thresholds tuned on dev.


In [1]:
import sys, json, traceback, time
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
while not (ROOT / ".gitignore").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
import torch
import lsi_utils as lu
from lsi_experiment import Experiment, compare
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 30)
print("project root:", ROOT, "| torch", torch.__version__, "| cuda", torch.cuda.is_available())

c:\Users\Atharva\Desktop\Course Sem 1\LLM\Capstone Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


project root: C:\Users\Atharva\Desktop\Course Sem 1\LLM\Capstone Project | torch 2.14.0+cu132 | cuda True


## 1. Data after the P5_01 decisions (cached tokenisation)


In [2]:
data, info = lu.load_lsi(ROOT)
for s, r in info.items():
    print(f"{s:5s}: {r}")

train: {'records': 42750, 'removed_duplicate_copies': 225, 'dropped_conflicting_groups': 80, 'labels_per_case': 3.698, 'kept': 42235}
dev  : {'records': 10181, 'labels_per_case': 3.916, 'kept': 10181}
test : {'records': 13019, 'kept': 13019}


## 2. The experiments

| #   | Phase-1 model        | technique                                                | why                                                       |
| --- | -------------------- | -------------------------------------------------------- | --------------------------------------------------------- |
| 1   | A2 decoder + copy    | **full fine-tuning**                                     | strongest model in Phases 1-4; also the **KD teacher**    |
| 2   | D encoder (4 layers) | **knowledge distillation** from #1 (mandatory)           | student = 58% of the parameters, 1024 tokens              |
| 2c  | D encoder            | same student **without** teacher (control, dev only)     | shows what KD adds                                        |
| 3   | B prefix-LM          | **LoRMA** r8 (compulsory)                                | multiplicative low-rank update, 1.5% of the weights       |
| 4   | A plain decoder      | **bottleneck adapters** (64)                             | new small layers, 2.3% of the weights                     |
| 5   | A2 decoder + copy    | **partial (layer-wise) fine-tuning**: top 4 of 10 blocks | cheaper than full FT: does the lower half need to change? |

`TEACHER` (logits of #1 on the training cases) is saved right after #1, so #2 can use it.


In [3]:
TEACHER = "A2_full_causal_first_last1536_attn"
EXPERIMENTS = [
    # (label, config, save teacher outputs?, make a test submission?)
    ("#1 A2 full FT (teacher)", {"phase1": "A2", "method": "full"}, True, True),
    ("#2 D encoder + KD",       {"phase1": "D", "method": "full", "kd_teacher": TEACHER}, False, True),
    ("#2c D encoder, no KD",    {"phase1": "D", "method": "full"}, False, False),
    ("#3 B + LoRMA r8",         {"phase1": "B", "method": "lorma"}, False, True),
    ("#4 A + adapters",         {"phase1": "A", "method": "adapter"}, False, True),
    ("#5 A2 partial FT top 4",  {"phase1": "A2", "method": "partial", "top_blocks": 4}, False, True),
]

def run(label, cfg, teacher, submit):
    print(f"\n{'=' * 100}\n{time.strftime('%Y-%m-%d %H:%M')}  {label}  {cfg}\n{'=' * 100}")
    exp = None
    try:
        exp = Experiment(ROOT, cfg)
        exp.memory_check()
        exp.train()
        exp.report()
        if submit:
            exp.predict_test()
        if teacher:
            exp.teacher_outputs()
        exp.speed()
        exp.log()
        print(f"{time.strftime('%H:%M')}  {label} finished")
    except Exception:
        print(f"{time.strftime('%H:%M')}  {label} FAILED:")
        traceback.print_exc(file=sys.stdout)
    finally:
        del exp
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

## 3. Run everything (each experiment prints its training progress, dev report and submission check)


In [4]:
for label, cfg, teacher, submit in EXPERIMENTS:
    run(label, cfg, teacher, submit)
print("\nall experiments done", time.strftime("%Y-%m-%d %H:%M"))


2026-10-25 22:20  #1 A2 full FT (teacher)  {'phase1': 'A2', 'method': 'full'}
train: {'records': 42750, 'removed_duplicate_copies': 225, 'dropped_conflicting_groups': 80, 'labels_per_case': 3.698, 'kept': 42235} | tokens median 652, max 141,120
dev  : {'records': 10181, 'labels_per_case': 3.916, 'kept': 10181} | tokens median 808, max 117,095
test : {'records': 13019, 'kept': 13019} | tokens median 798, max 222,314
experiment: A2_full_causal_first_last1536_attn | device cuda | precision torch.bfloat16
[cache] memory check: {'params_M': 70.12, 'trainable_M': 70.12, 'batch': '2 x 1536 tokens', 'sec': 1.21, 'peak_gb': 1.57, 'phase1_load': {'arch': 'A2', 'layers': 10, 'loaded_tensors': 72, 'dropped_tensors': 5, 'dropped_groups': ['copy_gate', 'copy_k', 'copy_q', 'lm_head']}}
parameters: 70.12M total, 70.12M trainable before freezing rules (100.0%)
[cache] training 'train' already done -> loaded best_train.pt (best epoch 3)
[cache] dev report loaded
dev @0.5                     : {'macro_f

## 4. Comparison (dev)

`dev_macro_f1_global` / `dev_macro_f1_per_label` are the honest **cross-fitted** dev scores of the two threshold
strategies (tuned on one half of dev, scored on the other half); `strategy` = the one used for the test submission.


In [5]:
tbl = compare(ROOT)
display(tbl)

,phase1,method,input,trainable_M_train,best_epoch,train_min_per_epoch,dev_macro_f1_at_0.5,dev_macro_f1_global,dev_macro_f1_per_label,strategy,dev_micro_f1,dev_subset_acc,cases_per_s,peak_gb
run,,,,,,,,,,,,,,
A2_full_causal_first_last1536_attn,A2,full,first_last1536,70.12,3,22.0,0.3847,0.3866,0.3881,per_label,0.4931,0.0479,98.3,0.62
B_lormar8_causal_first_last1536_attn,B,lorma,first_last1536,1.13,4,23.3,0.3733,0.3740,0.3799,per_label,0.4876,0.0419,67.0,0.57
A2_partial4_causal_first_last1536_attn,A2,partial,first_last1536,19.90,3,13.6,0.3675,0.3689,0.3774,per_label,0.4841,0.0433,100.2,0.56
A_adapter64_causal_first_last1536_attn,A,adapter,first_last1536,1.72,4,18.7,0.3670,0.3704,0.3753,per_label,0.4826,0.0429,92.6,0.53
D_full_bidir_first_last1024_attn_kd2,D,full,first_last1024,40.38,3,10.8,0.3565,0.3561,0.3581,per_label,0.4666,0.0326,144.8,0.44
D_full_bidir_first_last1024_attn,D,full,first_last1024,40.38,3,10.8,0.3350,0.3346,0.3396,per_label,0.4460,0.0292,120.1,0.44


## 5. Knowledge distillation: what does the student keep, and how much cheaper is it?


In [6]:
rows = {r["run"]: r for r in lu.read_experiments(ROOT)}
kd_runs = [TEACHER, "D_full_bidir_first_last1024_attn_kd2", "D_full_bidir_first_last1024_attn"]
kd = pd.DataFrame([rows[r] for r in kd_runs if r in rows]).set_index("run")
if len(kd):
    cols = [c for c in ["params_M", "dev_macro_f1_per_label", "dev_micro_f1", "cases_per_s", "peak_gb", "train_min_per_epoch"] if c in kd]
    display(kd[cols])

,params_M,dev_macro_f1_per_label,dev_micro_f1,cases_per_s,peak_gb,train_min_per_epoch
run,,,,,,
A2_full_causal_first_last1536_attn,70.12,0.3881,0.4931,98.3,0.62,22.0
D_full_bidir_first_last1024_attn_kd2,40.38,0.3581,0.4666,144.8,0.44,10.8
D_full_bidir_first_last1024_attn,40.38,0.3396,0.4460,120.1,0.44,10.8


## 6. Rare statutes and threshold strategies per experiment


In [7]:
for r in rows:
    p = ROOT / "checkpoints" / "phase5" / r / "dev_report.json"
    if p.exists():
        rep = json.load(open(p))
        print(f"{r}\n   global {rep['cross_fitted']['global']['macro_f1']} vs per statute {rep['cross_fitted']['per_label']['macro_f1']}"
              f" | by frequency: {[(b['bucket'], b['mean_f1']) for b in rep['f1_by_frequency']]}")

A2_full_causal_first_last1536_attn
   global 0.3866 vs per statute 0.3881 | by frequency: [('>=1000 train cases', 0.4889), ('300-999', 0.3467), ('<300', 0.2164)]
D_full_bidir_first_last1024_attn_kd2
   global 0.3561 vs per statute 0.3581 | by frequency: [('>=1000 train cases', 0.4659), ('300-999', 0.3107), ('<300', 0.1974)]
D_full_bidir_first_last1024_attn
   global 0.3346 vs per statute 0.3396 | by frequency: [('>=1000 train cases', 0.4458), ('300-999', 0.2928), ('<300', 0.1653)]
B_lormar8_causal_first_last1536_attn
   global 0.374 vs per statute 0.3799 | by frequency: [('>=1000 train cases', 0.4832), ('300-999', 0.3345), ('<300', 0.2266)]
A_adapter64_causal_first_last1536_attn
   global 0.3704 vs per statute 0.3753 | by frequency: [('>=1000 train cases', 0.4814), ('300-999', 0.3307), ('<300', 0.2237)]
A2_partial4_causal_first_last1536_attn
   global 0.3689 vs per statute 0.3774 | by frequency: [('>=1000 train cases', 0.4775), ('300-999', 0.3345), ('<300', 0.2345)]


## 7. Experiment #6 - hierarchical whole-case model on the frozen #1 (`src/lsi_hier.py`)

The whole case is cut into chunks of 1536 tokens (max 16; longer cases: first 8 + last 8). The FROZEN #1 model turns every
chunk into a vector + its own 100 statute scores. A small head (BiGRU over chunks + **one attention over the chunks per
statute** + skip connection from "max over chunks") is trained on **dev with 5-fold cross-validation x 3 seeds**
(the encoder never saw dev, so no stacking leakage); test = average of the 15 heads. ~10-25 min, everything cached.

Also printed: training-free baselines (first chunk / max / mean over chunks), section ablation (first / last / first+last /
all chunks), one shared attention vs one per statute, #1 vs #6 by case length, attention by chunk position.


In [8]:
import lsi_hier as lh
enc = Experiment(ROOT, {"phase1": "A2", "method": "full"})     # experiment #1 (loaded from its run folder)
hier_rep = lh.run_hier(ROOT, enc)
del enc
display(compare(ROOT))

experiment: A2_full_causal_first_last1536_attn | device cuda | precision torch.bfloat16
parameters: 70.12M total, 70.12M trainable before freezing rules (100.0%)
[cache] training 'train' already done -> loaded best_train.pt (best epoch 3)


encoding test chunks: 100%|██████████| 1906/1906 [04:11<00:00,  7.58it/s]


dev: 10181 cases, 18609 chunks (mean 1.83, max 16) | test: 23090 chunks
training-free baselines (dev, cross-fitted macro-F1): {'first chunk only (frozen #1)': 0.3777, 'max over chunks': 0.3851, 'mean over chunks': 0.3799}


section ablation: 100%|██████████| 4/4 [05:31<00:00, 82.80s/it]


ablations (dev, cross-fitted macro-F1, seed 0): {'chunks: first': 0.3653, 'chunks: last': 0.3426, 'chunks: first+last': 0.3689, 'chunks: all': 0.3721, 'all chunks, ONE shared attention': 0.3738}


final heads (seeds): 100%|██████████| 3/3 [05:40<00:00, 113.36s/it]


[cache] dev report loaded
dev @0.5                     : {'macro_f1': 0.3847, 'micro_f1': 0.4793, 'micro_p': 0.4284, 'micro_r': 0.5439, 'macro_p': 0.3429, 'macro_r': 0.4507, 'subset_acc': 0.0442, 'labels_per_case': 4.972}
dev global threshold 0.58   : {'macro_f1': 0.3871, 'micro_f1': 0.4814, 'micro_p': 0.4735, 'micro_r': 0.4896, 'macro_p': 0.3768, 'macro_r': 0.4078, 'subset_acc': 0.054, 'labels_per_case': 4.049}
dev per-statute thresholds   : {'macro_f1': 0.4052, 'micro_f1': 0.4931, 'micro_p': 0.4399, 'micro_r': 0.561, 'macro_p': 0.3953, 'macro_r': 0.4367, 'subset_acc': 0.0479, 'labels_per_case': 4.994}  (tuned and scored on the same cases)
cross-fitted (honest) macro-F1: global 0.3866 | per statute 0.3881 -> using per_label
F1 by statute frequency: [{'bucket': '>=1000 train cases', 'statutes': 43, 'mean_f1': 0.4889}, {'bucket': '300-999', 'statutes': 55, 'mean_f1': 0.3467}, {'bucket': '<300', 'statutes': 2, 'mean_f1': 0.2164}]
worst statutes: [(8, 0.1029), (2, 0.1224), (0, 0.1244), (3

,phase1,method,input,trainable_M_train,best_epoch,train_min_per_epoch,dev_macro_f1_at_0.5,dev_macro_f1_global,dev_macro_f1_per_label,strategy,dev_micro_f1,dev_subset_acc,cases_per_s,peak_gb
run,,,,,,,,,,,,,,
A2_hier_frozen,A2,frozen #1 + hierarchical head (dev CV),whole case (<= 16 x 1536),0.335,NaN,NaN,NaN,0.3891,0.3899,per_label,0.4985,0.0523,NaN,NaN
A2_full_causal_first_last1536_attn,A2,full,first_last1536,70.120,3.0,22.0,0.3847,0.3866,0.3881,per_label,0.4931,0.0479,98.3,0.62
B_lormar8_causal_first_last1536_attn,B,lorma,first_last1536,1.130,4.0,23.3,0.3733,0.3740,0.3799,per_label,0.4876,0.0419,67.0,0.57
A2_partial4_causal_first_last1536_attn,A2,partial,first_last1536,19.900,3.0,13.6,0.3675,0.3689,0.3774,per_label,0.4841,0.0433,100.2,0.56
A_adapter64_causal_first_last1536_attn,A,adapter,first_last1536,1.720,4.0,18.7,0.3670,0.3704,0.3753,per_label,0.4826,0.0429,92.6,0.53
D_full_bidir_first_last1024_attn_kd2,D,full,first_last1024,40.380,3.0,10.8,0.3565,0.3561,0.3581,per_label,0.4666,0.0326,144.8,0.44
D_full_bidir_first_last1024_attn,D,full,first_last1024,40.380,3.0,10.8,0.3350,0.3346,0.3396,per_label,0.4460,0.0292,120.1,0.44
